In [2]:
# Install dependencies
!pip install torch torchvision --index-url https://pytorch.org
!pip install diffusers transformers accelerate safetensors pillow

Looking in indexes: https://pytorch.org


In [3]:
!nvidia-smi

Mon Aug 24 16:03:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

Mounted at /content/drive


In [5]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/generative_rockets/

total 14899
-rw------- 1 root root  191003 Aug 19 21:11 10.png
-rw------- 1 root root 1074311 Aug 22 22:18 11.png
-rw------- 1 root root  574114 Aug 22 22:20 12.png
-rw------- 1 root root  505267 Aug 22 22:22 13.png
-rw------- 1 root root  606461 Aug 22 22:23 14.png
-rw------- 1 root root  226947 Aug 22 22:23 15.png
-rw------- 1 root root  825329 Aug 22 22:25 16.png
-rw------- 1 root root  230436 Aug 24 15:54 17.png
-rw------- 1 root root  432220 Aug 24 15:55 18.png
-rw------- 1 root root  959075 Aug 24 15:56 19.png
-rw------- 1 root root 2071496 Aug 19 21:02 1.png
-rw------- 1 root root  658673 Aug 24 15:57 20.png
-rw------- 1 root root 1990696 Aug 24 15:57 21.png
-rw------- 1 root root  809303 Aug 24 15:57 22.png
-rw------- 1 root root  356262 Aug 24 15:58 23.png
-rw------- 1 root root  393754 Aug 24 16:00 24.png
-rw------- 1 root root  409059 Aug 19 21:02 2.png
-rw------- 1 root root  401250 Aug 19 21:03 3.png
-rw------- 1 root root  509367 Aug 19 21:06 4.png
-rw------- 1 root root 

In [6]:
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import os
import gc
import random

assert torch.cuda.is_available(), "CUDA is required to run SDXL locally."

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [7]:
# Makes sure there are not more pipelines running in background
if "pipeline" in globals():
    del pipeline

# downloads model from Hugging Face cache
pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

# downloads the adapter subfolder weights
pipeline.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter_sdxl.bin"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/2.01k [00:00<?, ?B/s]

sdxl_models/image_encoder/model.safetens(…): reconstructing file:   0%|          |  0.00B / 3.69GB            

sdxl_models/image_encoder/model.safetens(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [46]:
# Set strength scale (0.5 to 0.8 balances image influence vs text prompt)
pipeline.set_ip_adapter_scale(0.6)

# Force the VAE decoder to match the float16 pipeline precision
pipeline.vae.to(dtype=torch.float16)

# Setting up memory saving configurations
pipeline.enable_freeu(s1=0.9, s2=0.2, b1=1.3, b2=1.4)
pipeline.unet.to(memory_format=torch.channels_last) 
pipeline.vae.enable_slicing()
pipeline.vae.enable_tiling()

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [47]:
# Forces Python's Garbage Collector to scan and destroy unlinked objects
gc.collect()

# Empty PyTorch's internal GPU memory cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
image_dir = "/content/drive/MyDrive/generative_rockets/" 
image_paths = random.sample([os.path.join(image_dir, f) for f in sorted(os.listdir(image_dir)) if f.endswith(('png', 'jpg', 'jpeg'))],15)
print(f"Image paths: ", image_paths)
output_path = "generated_fusion_spaceship.png"

reference_images = []

for p in image_paths:
    img = Image.open(p)
    img_rgb = img.convert("RGB")
    reference_images.append(img_rgb)

prompt = (
    "A spaceship far away. The spaceship is human made. Make sure to show the whole spaceship"
    "combining structural design cues from the reference images, "
    "we should see the engines"
    "intricate metal armor plating, glowing purple and yellow ion engines, it has a recolector of dark martter in the front,"
    "the engines are in the back and the sides, 8k resolution"
)

print("Generating image...")

generator = torch.Generator(device="cuda").manual_seed(1720)
generated_image = pipeline(
    prompt=prompt,
    ip_adapter_image=[reference_images],
    num_inference_steps=50,
    generator=generator,
).images[0]


generated_image.save(output_path)
print(f"Saved generated spaceship to {output_path}")


Image paths:  ['/content/drive/MyDrive/generative_rockets/1.png', '/content/drive/MyDrive/generative_rockets/29.png', '/content/drive/MyDrive/generative_rockets/9.png', '/content/drive/MyDrive/generative_rockets/17.png', '/content/drive/MyDrive/generative_rockets/34.png', '/content/drive/MyDrive/generative_rockets/19.png', '/content/drive/MyDrive/generative_rockets/26.png', '/content/drive/MyDrive/generative_rockets/30.png', '/content/drive/MyDrive/generative_rockets/15.png', '/content/drive/MyDrive/generative_rockets/23.png', '/content/drive/MyDrive/generative_rockets/36.png', '/content/drive/MyDrive/generative_rockets/18.png', '/content/drive/MyDrive/generative_rockets/24.png', '/content/drive/MyDrive/generative_rockets/8.png', '/content/drive/MyDrive/generative_rockets/31.png']
Generating image...


  0%|          | 0/50 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Saved generated spaceship to generated_fusion_spaceship.png
